# Demo 2: Batch ETL/ELT pipeline DuckDB-vel

BME Adatmérnökség – 2. hét

Ebben a demóban:
1. Extract: PostgreSQL → Parquet (full vs incremental)
2. Transform: DuckDB SQL transzformációk (staging → cleaned → aggregated)
3. SCD Type 2 implementáció
4. Load: Gold réteg összesítő táblák

In [1]:
import psycopg2
import psycopg2.extras
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import os
import time
from datetime import datetime, timedelta

# Könyvtárak létrehozása
os.makedirs("/home/jovyan/data/raw", exist_ok=True)
os.makedirs("/home/jovyan/data/staging", exist_ok=True)
os.makedirs("/home/jovyan/data/gold", exist_ok=True)

# PostgreSQL kapcsolat
pg_conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="webshop", user="dataeng", password="dataeng2024"
)
pg_cur = pg_conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# DuckDB in-memory
duck = duckdb.connect()

print("Kapcsolatok létrehozva!")
print(f"  PostgreSQL: webshop")
print(f"  DuckDB: in-memory")

Kapcsolatok létrehozva!
  PostgreSQL: webshop
  DuckDB: in-memory


## 1. Extract: Full Snapshot

Az összes adat kinyerése PostgreSQL-ből Parquet fájlokba.

In [2]:
# Full Extract - minden tábla → Parquet
tables = ['customers', 'products', 'orders', 'order_items']

print("=== Full Extract: PostgreSQL → Parquet ===\n")
start = time.time()

for table in tables:
    pg_cur.execute(f"SELECT * FROM {table}")
    rows = pg_cur.fetchall()
    
    if not rows:
        print(f"  {table}: üres tábla, kihagyva")
        continue
    
    # Dict list → PyArrow Table → Parquet
    columns = {k: [r[k] for r in rows] for k in rows[0].keys()}
    arrow_table = pa.table(columns)
    
    path = f"/home/jovyan/data/raw/{table}.parquet"
    pq.write_table(arrow_table, path)
    
    file_size = os.path.getsize(path) / 1024
    print(f"  {table:15s} → {len(rows):>5} sor, {file_size:.1f} KB")

elapsed = time.time() - start
print(f"\nFull extract kész: {elapsed:.2f} mp")

=== Full Extract: PostgreSQL → Parquet ===

  customers       →    30 sor, 4.1 KB
  products        →    25 sor, 3.6 KB
  orders          →   100 sor, 4.5 KB
  order_items     →   200 sor, 4.2 KB

Full extract kész: 0.42 mp


## 2. Extract: Incremental (összehasonlítás)

Csak az utolsó 7 napban módosult rekordok kinyerése.

In [3]:
# Incremental Extract - csak az utolsó 7 nap
print("=== Incremental Extract (utolsó 7 nap) ===\n")
start = time.time()

cutoff = datetime.now() - timedelta(days=7)

# Orders - incremental by order_date
pg_cur.execute("SELECT * FROM orders WHERE order_date >= %s", (cutoff,))
rows = pg_cur.fetchall()
if rows:
    columns = {k: [r[k] for r in rows] for k in rows[0].keys()}
    arrow_table = pa.table(columns)
    path = "/home/jovyan/data/raw/orders_incremental.parquet"
    pq.write_table(arrow_table, path)
    file_size = os.path.getsize(path) / 1024
    print(f"  orders (incremental): {len(rows)} sor, {file_size:.1f} KB")

# Full vs Incremental összehasonlítás
full_size = os.path.getsize("/home/jovyan/data/raw/orders.parquet")
incr_size = os.path.getsize(path) if rows else 0
pg_cur.execute("SELECT COUNT(*) as cnt FROM orders")
total = pg_cur.fetchone()['cnt']

elapsed = time.time() - start
print(f"\n--- Összehasonlítás ---")
print(f"  Full extract:        {total} sor, {full_size/1024:.1f} KB")
print(f"  Incremental extract: {len(rows) if rows else 0} sor, {incr_size/1024:.1f} KB")
print(f"  Incremental idő:    {elapsed:.3f} mp")
print(f"  Adatcsökkentés:     {(1 - incr_size/full_size)*100:.0f}%")

=== Incremental Extract (utolsó 7 nap) ===

  orders (incremental): 12 sor, 2.8 KB

--- Összehasonlítás ---
  Full extract:        100 sor, 4.5 KB
  Incremental extract: 12 sor, 2.8 KB
  Incremental idő:    0.038 mp
  Adatcsökkentés:     38%


## 3. Transform: Staging réteg

DuckDB-vel olvassuk be a Parquet fájlokat és létrehozzuk a staging táblákat.

In [4]:
# Staging réteg: Parquet → DuckDB táblák
print("=== Staging réteg létrehozása ===\n")

for table in tables:
    path = f"/home/jovyan/data/raw/{table}.parquet"
    duck.execute(f"CREATE OR REPLACE TABLE stg_{table} AS SELECT * FROM read_parquet('{path}')")
    cnt = duck.execute(f"SELECT COUNT(*) FROM stg_{table}").fetchone()[0]
    print(f"  stg_{table:15s} → {cnt} sor")

# Séma ellenőrzés
print("\n=== stg_orders séma ===")
cols = duck.execute("DESCRIBE stg_orders").fetchall()
for col in cols:
    print(f"  {col[0]:20s} {col[1]}")

=== Staging réteg létrehozása ===

  stg_customers       → 30 sor
  stg_products        → 25 sor
  stg_orders          → 100 sor
  stg_order_items     → 200 sor

=== stg_orders séma ===
  order_id             BIGINT
  customer_id          BIGINT
  order_date           TIMESTAMP
  status               VARCHAR
  total_amount         DECIMAL(9,2)
  shipping_city        VARCHAR


## 4. Transform: Tisztítás és üzleti logika

In [5]:
# Tisztított és gazdagított rendelések
print("=== Transzformációk ===\n")

# 1. Cleaned orders: csak érvényes rendelések + ügyfélnév
duck.execute("""
    CREATE OR REPLACE TABLE cleaned_orders AS
    SELECT 
        o.order_id,
        o.customer_id,
        c.name as customer_name,
        c.segment as customer_segment,
        o.order_date,
        o.status,
        o.total_amount,
        o.shipping_city,
        CASE 
            WHEN o.total_amount >= 200000 THEN 'nagy'
            WHEN o.total_amount >= 50000 THEN 'közepes'
            ELSE 'kis'
        END as order_size
    FROM stg_orders o
    JOIN stg_customers c ON o.customer_id = c.customer_id
    WHERE o.status != 'cancelled'
""")
cnt = duck.execute("SELECT COUNT(*) FROM cleaned_orders").fetchone()[0]
print(f"  cleaned_orders: {cnt} sor (cancelled kiszűrve)")

# 2. Napi aggregáció
duck.execute("""
    CREATE OR REPLACE TABLE daily_sales AS
    SELECT 
        CAST(order_date AS DATE) as order_day,
        COUNT(*) as order_count,
        SUM(total_amount) as total_revenue,
        AVG(total_amount)::NUMERIC(12,2) as avg_order_value,
        COUNT(DISTINCT customer_id) as unique_customers
    FROM cleaned_orders
    GROUP BY CAST(order_date AS DATE)
    ORDER BY order_day
""")
cnt = duck.execute("SELECT COUNT(*) FROM daily_sales").fetchone()[0]
print(f"  daily_sales: {cnt} nap")

# 3. Kategória statisztikák
duck.execute("""
    CREATE OR REPLACE TABLE category_stats AS
    SELECT 
        p.category,
        COUNT(DISTINCT oi.order_id) as order_count,
        SUM(oi.quantity) as total_quantity,
        SUM(oi.quantity * oi.unit_price)::NUMERIC(12,2) as total_revenue,
        AVG(oi.unit_price)::NUMERIC(12,2) as avg_price
    FROM stg_order_items oi
    JOIN stg_products p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
""")
print("\n=== Kategória statisztikák ===")
rows = duck.execute("SELECT * FROM category_stats").fetchall()
print(f"  {'Kategória':20s} {'Rendelések':>10s} {'Bevétel':>15s}")
print(f"  {'-'*45}")
for r in rows:
    print(f"  {r[0]:20s} {r[1]:>10d} {r[3]:>15,.0f} Ft")

=== Transzformációk ===

  cleaned_orders: 86 sor (cancelled kiszűrve)
  daily_sales: 57 nap

=== Kategória statisztikák ===
  Kategória            Rendelések         Bevétel
  ---------------------------------------------
  Háztartás                    28      10,689,420 Ft
  TV                            7       8,499,830 Ft
  Monitor                      16       6,839,640 Ft
  Mobiltelefon                  9       6,299,820 Ft
  Fotó                         10       6,299,790 Ft
  Audio                        21       5,154,520 Ft
  Elektronika                  10       3,239,800 Ft
  Viselhető                    13       2,984,720 Ft
  Tablet                        3       1,679,940 Ft
  Játék                        11       1,654,800 Ft
  Periféria                    14       1,389,720 Ft
  E-könyvolvasó                13       1,299,740 Ft
  Okosotthon                    9         629,820 Ft
  Tárhely                       9         594,810 Ft
  Hálózat                       7  

## 5. SCD Type 2: Ügyfél dimenzió

Slowly Changing Dimension Type 2: teljes történetiség megőrzése.

In [6]:
# SCD Type 2 implementáció
print("=== SCD Type 2: Ügyfél dimenzió ===\n")

# Kezdeti betöltés - minden ügyfél aktuális
duck.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd2 AS
    SELECT 
        customer_id,
        name,
        city,
        segment,
        created_at as valid_from,
        CAST('9999-12-31' AS TIMESTAMP) as valid_to,
        TRUE as is_current,
        1 as version
    FROM stg_customers
""")
cnt = duck.execute("SELECT COUNT(*) FROM dim_customer_scd2").fetchone()[0]
print(f"Kezdeti betöltés: {cnt} ügyfél (mind is_current=true)\n")

# Szimuláljuk egy ügyfél változását
print("Szimuláció: 3 ügyfél szegmenst vált...")
changes = [
    (1, 'gold', 'Székesfehérvár'),
    (5, 'premium', 'Budapest'),
    (10, 'gold', 'Pécs'),
]

now = datetime.now()
for cust_id, new_segment, new_city in changes:
    # Régi rekord lezárása
    duck.execute(f"""
        UPDATE dim_customer_scd2 
        SET valid_to = CAST('{now}' AS TIMESTAMP), is_current = FALSE
        WHERE customer_id = {cust_id} AND is_current = TRUE
    """)
    
    # Új verzió beszúrása
    duck.execute(f"""
        INSERT INTO dim_customer_scd2 
        SELECT 
            customer_id, name, '{new_city}', '{new_segment}',
            CAST('{now}' AS TIMESTAMP), CAST('9999-12-31' AS TIMESTAMP), TRUE,
            version + 1
        FROM dim_customer_scd2 
        WHERE customer_id = {cust_id} AND is_current = FALSE
        ORDER BY version DESC LIMIT 1
    """)
    print(f"  Ügyfél #{cust_id}: segment → {new_segment}, city → {new_city}")

# Eredmény – DuckDB szintaxissal: strftime a timestamp formázáshoz
print(f"\n=== SCD2 tábla: változott ügyfelek ===\n")
rows = duck.execute("""
    SELECT customer_id, name, city, segment, 
           strftime(valid_from, '%Y-%m-%d %H:%M:%S') as valid_from,
           CASE WHEN valid_to > CAST('9998-01-01' AS TIMESTAMP)
                THEN 'aktuális'
                ELSE strftime(valid_to, '%Y-%m-%d %H:%M:%S')
           END as valid_to,
           is_current, version
    FROM dim_customer_scd2 
    WHERE customer_id IN (1, 5, 10)
    ORDER BY customer_id, version
""").fetchall()
print(f"  {'ID':>3s} {'Név':20s} {'Város':15s} {'Szegmens':10s} {'Verzió':>6s} {'Aktuális':>8s}")
for r in rows:
    print(f"  {r[0]:>3d} {r[1]:20s} {r[2]:15s} {r[3]:10s} {r[7]:>6d} {'igen' if r[6] else 'nem':>8s}")

=== SCD Type 2: Ügyfél dimenzió ===

Kezdeti betöltés: 30 ügyfél (mind is_current=true)

Szimuláció: 3 ügyfél szegmenst vált...
  Ügyfél #1: segment → gold, city → Székesfehérvár
  Ügyfél #5: segment → premium, city → Budapest
  Ügyfél #10: segment → gold, city → Pécs

=== SCD2 tábla: változott ügyfelek ===

   ID Név                  Város           Szegmens   Verzió Aktuális
    1 Kovács Anna          Budapest        premium         1      nem
    1 Kovács Anna          Székesfehérvár  gold            2     igen
    5 Horváth Gábor        Győr            standard        1      nem
    5 Horváth Gábor        Budapest        premium         2     igen
   10 Farkas Judit         Szeged          standard        1      nem
   10 Farkas Judit         Pécs            gold            2     igen


## 6. Load: Gold réteg → Parquet export

In [7]:
# Gold réteg exportálás
print("=== Gold réteg: DuckDB → Parquet ===\n")

gold_tables = {
    'cleaned_orders': "/home/jovyan/data/gold/cleaned_orders.parquet",
    'daily_sales': "/home/jovyan/data/gold/daily_sales.parquet",
    'category_stats': "/home/jovyan/data/gold/category_stats.parquet",
    'dim_customer_scd2': "/home/jovyan/data/gold/dim_customer_scd2.parquet",
}

for table, path in gold_tables.items():
    duck.execute(f"COPY {table} TO '{path}' (FORMAT PARQUET)")
    file_size = os.path.getsize(path) / 1024
    cnt = duck.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:25s} → {cnt:>5} sor, {file_size:.1f} KB")

# Parquet metaadatok ellenőrzése
print("\n=== Parquet metaadatok (cleaned_orders) ===")
pf = pq.read_metadata("/home/jovyan/data/gold/cleaned_orders.parquet")
print(f"  Sorok: {pf.num_rows}")
print(f"  Oszlopok: {pf.num_columns}")
print(f"  Row group-ok: {pf.num_row_groups}")
print(f"  Méret: {pf.serialized_size / 1024:.1f} KB")

=== Gold réteg: DuckDB → Parquet ===

  cleaned_orders            →    86 sor, 4.1 KB
  daily_sales               →    57 sor, 2.0 KB
  category_stats            →    16 sor, 1.2 KB
  dim_customer_scd2         →    33 sor, 2.3 KB

=== Parquet metaadatok (cleaned_orders) ===
  Sorok: 86
  Oszlopok: 9
  Row group-ok: 1
  Méret: 1.0 KB


In [8]:
# Teljesítmény összehasonlítás: PostgreSQL vs DuckDB Parquet
print("=== Teljesítmény összehasonlítás ===\n")

# PostgreSQL lekérdezés
start = time.time()
pg_cur.execute("""
    SELECT p.category, COUNT(*) as cnt, SUM(oi.quantity * oi.unit_price) as revenue
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category ORDER BY revenue DESC
""")
pg_cur.fetchall()
pg_time = time.time() - start

# DuckDB Parquet lekérdezés
start = time.time()
duck.execute("""
    SELECT p.category, COUNT(*) as cnt, SUM(oi.quantity * oi.unit_price) as revenue
    FROM read_parquet('/home/jovyan/data/raw/order_items.parquet') oi 
    JOIN read_parquet('/home/jovyan/data/raw/products.parquet') p ON oi.product_id = p.product_id
    GROUP BY p.category ORDER BY revenue DESC
""").fetchall()
duck_time = time.time() - start

print(f"  PostgreSQL:      {pg_time*1000:.1f} ms")
print(f"  DuckDB (Parquet): {duck_time*1000:.1f} ms")
print(f"\nMegjegyzés: kis adathalmazon a különbség minimális.")
print("Nagy adatnál (millió+ sor) a DuckDB columnar előnye jelentős!")

=== Teljesítmény összehasonlítás ===

  PostgreSQL:      7.0 ms
  DuckDB (Parquet): 37.3 ms

Megjegyzés: kis adathalmazon a különbség minimális.
Nagy adatnál (millió+ sor) a DuckDB columnar előnye jelentős!


In [9]:
# Takarítás
duck.close()
pg_cur.close()
pg_conn.close()

print("=== Demo 2 összefoglalás ===")
print("1. Full Extract: PostgreSQL → Parquet (minden tábla)")
print("2. Incremental Extract: csak a friss adatok (kevesebb I/O)")
print("3. Staging → Cleaned → Aggregated transzformáció DuckDB-vel")
print("4. SCD Type 2: ügyfél dimenzió történetiség")
print("5. Gold réteg: Parquet export analitikához")
print("\nKövetkező: Demo 3 – Kafka streaming pipeline")

=== Demo 2 összefoglalás ===
1. Full Extract: PostgreSQL → Parquet (minden tábla)
2. Incremental Extract: csak a friss adatok (kevesebb I/O)
3. Staging → Cleaned → Aggregated transzformáció DuckDB-vel
4. SCD Type 2: ügyfél dimenzió történetiség
5. Gold réteg: Parquet export analitikához

Következő: Demo 3 – Kafka streaming pipeline
